In [ ]:
import ipywidgets as widgets
import torch
from pathlib import Path
from safetensors.torch import load_file
from transformers import AutoTokenizer
import random

class CacheLoader:
    def __init__(self, cache_directory, tokenizer):
        self.cache_directory = Path(cache_directory)
        self.files = sorted(list(self.cache_directory.glob("*.safetensors")))
        self.file_ranges = []
        for file in self.files:
            range = file.stem.split("_")
            self.file_ranges.append({"path": file, "start": int(range[0]), "end": int(range[1])})
        self.tokenizer = tokenizer

    def has_sufficient_data(self, feature_id, minimum_count):
        file = [file for file in self.file_ranges if file["start"] <= feature_id <= file["end"]][0]
        cache = load_file(file["path"])        
        locations = cache["locations"].to(torch.int64)
        local_feature_id = feature_id - file["start"]
        count = (locations[:, 2] == local_feature_id).sum().item()
        return count >= minimum_count

    def _get_activating_examples(self, feature_id):
        file = [f for f in self.file_ranges if f["start"] <= feature_id <= f["end"]][0]
        cache = load_file(file["path"])
        locations = cache["locations"].to(torch.int64) 
        local_feature_id = feature_id - file["start"]
        mask = locations[:, 2] == local_feature_id        
        activations = cache["activations"][mask]
        batch_indices = locations[mask, 0]
        activating_examples = []
        for activation, batch_index in zip(activations, batch_indices):
            activating_examples.append({
                "score": activation.item(),
                "batch_index": batch_index.item()
            })
        return file, activating_examples
    
    def _clean_text(self, text):
        text = text.replace("<|begin_of_text|>", "")
        text = text.replace("<|start_header_id|>", "\n\n")
        text = text.replace("<|end_header_id|>", "")
        text = text.replace("<|eot_id|>", "") 
        return text.strip()

    def _decode_activating_examples(self, file, activating_examples):
        cache = load_file(file["path"])
        activating_examples_text = []
        for activating_example in activating_examples:
            score = activating_example["score"]
            batch_index = activating_example["batch_index"]
            chosen_tokens = cache["tokens"][batch_index]
            rejected_tokens = cache["tokens_rejected"][batch_index]
            raw_chosen_text = self.tokenizer.decode([token for token in chosen_tokens if token != self.tokenizer.pad_token_id], skip_special_tokens=False)
            raw_rejected_text = self.tokenizer.decode([token for token in rejected_tokens if token != self.tokenizer.pad_token_id], skip_special_tokens=False)
            chosen_text = self._clean_text(raw_chosen_text)
            rejected_text = self._clean_text(raw_rejected_text)
            activating_examples_text.append({
                "score": score,
                "chosen": chosen_text,
                "rejected": rejected_text,
            })
        return activating_examples_text

    def _split_held_out_set(self, activating_examples, held_out_count, seed):
        rng = random.Random(seed)
        held_out_set = rng.sample(activating_examples, held_out_count)
        held_out_set_keys = {activating_example["batch_index"] for activating_example in held_out_set}
        training_set = [activating_example for activating_example in activating_examples if activating_example["batch_index"] not in held_out_set_keys]
        return training_set, held_out_set

    def get_top_examples(
        self,
        feature_id,
        k,
        held_out_count,
        seed,
        random_heldouts=False,
    ):
        file, activating_examples = self._get_activating_examples(feature_id)
        activating_examples.sort(key=lambda x: x["score"])
        bottom_k = activating_examples[:k]
        top_k = activating_examples[-k:]
        training_set = bottom_k + top_k
        decoded_training_set = self._decode_activating_examples(file, training_set)
        if held_out_count == 0:
            return decoded_training_set
        training_indices = {activating_example["batch_index"] for activating_example in training_set}
        remaining_pool = [
            activating_example for activating_example in activating_examples 
            if activating_example["batch_index"] not in training_indices
        ]
        if random_heldouts:      
            rng = random.Random(seed)
            held_out_set = rng.sample(remaining_pool, held_out_count)
        else:
            held_out_set = remaining_pool[:held_out_count // 2] + remaining_pool[-(held_out_count // 2):]
        decoded_held_out_set = self._decode_activating_examples(file, held_out_set)
        return decoded_training_set, decoded_held_out_set

    def get_quantile_examples(
        self,
        feature_id,
        quantile_count,
        samples_per_quantile,
        held_out_count,
        seed,
    ):
        file, activating_examples = self._get_activating_examples(feature_id)
        if held_out_count == 0:
            training_set = activating_examples
        else:
            training_set, held_out_set = self._split_held_out_set(activating_examples, held_out_count, seed)
        training_set.sort(key=lambda x: x["score"])
        total = len(training_set)
        sampled_activating_examples = []
        rng = random.Random(seed)
        for quantile in range(quantile_count):
            start_index = int(quantile * (total / quantile_count))
            end_index = int((quantile + 1) * (total / quantile_count))
            if start_index == end_index:
                bucket = [training_set[start_index]] 
            else:
                bucket = training_set[start_index:end_index]
            sample = rng.sample(bucket, samples_per_quantile)
            sample.sort(key=lambda x: x["score"])
            for activating_example in sample:
                sampled_activating_examples.append(activating_example)
        decoded_samples = self._decode_activating_examples(file, sampled_activating_examples)
        if held_out_count == 0:
            return decoded_samples
        decoded_held_out = self._decode_activating_examples(file, held_out_set)
        return decoded_samples, decoded_held_out


# --------------------------------------------------


CACHE_DIRECTORY = "results/hhrlhf_diff_32/norm" 
MODEL_ID = "Skywork/Skywork-Reward-V2-Llama-3.1-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None: tokenizer.pad_token_id = tokenizer.eos_token_id

loader = CacheLoader(CACHE_DIRECTORY, tokenizer)

In [ ]:
labeling_instructions = """You are given a list of response pairs, where each response pair defines two possible responses to some prompt. Across the entire list of response pairs, there exists a feature that consistently distinguishes response one from response two.
Associated with each response pair is a score:
A positive score means the feature is more pronounced in response one.
A negative score means the feature is more pronounced in response two.



Your task:
Identify the feature that consistently distinguishes response one from response two.



Provide:
- A short, descriptive label for the feature.
- A sentence-long explanation of the difference.



Example response pair:

Positive Example 1: 
Response One:
user
Write a text message congratulating my friend on their new job.
assistant
Omg congratulations!! 🎉 That is such amazing news! 🥳 You are going to crush it! 💪✨
Response Two:
user
Write a text message congratulating my friend on their new job.
assistant
Congratulations on the new position. That is excellent news. I am sure you will perform very well.

Negative Example 1:
Response One:
user
How does this dress look?
assistant
It looks very nice on you. The color suits you well.
Response Two:
user
How does this dress look?
assistant
It looks stunning! 👗 The color is perfect on you! 😍🔥 10/10!

Example response pair label and explanation:
Label: Emoji usage
Explanation: When the score is positive, emojis are used more frequently in response one and when the score is negative, emojis are used more frequently in response two.



Output format (follow exactly, do not output any extra text):
Label: <short, descriptive label>
Explanation:  When the score is positive, <explain how the feature is more pronounced in response one> and when the score is negative, <explain how the feature is more pronounced in response two>.



Given these instructions, identify the feature present in the following response pairs:



"""

In [ ]:
prediction_instructions = """You are given a label and an explanation describing a feature that distinguishes two different responses to a prompt.


Your task:
Determine whether the score is positive or negative for the two different responses to the prompt.



Provide:
A single word indicating the predicted score: Positive or Negative.



Example label, explanation, and response pair:

Label: Emoji usage
Explanation: When the score is positive, emojis are used more frequently in response one and when the score is negative, emojis are used more frequently in response two.

Response One:
user
Write a text message congratulating my friend on their new job.
assistant
Omg congratulations!! 🎉 That is such amazing news! 🥳 You are going to crush it! 💪✨
Response Two:
user
Write a text message congratulating my friend on their new job.
assistant
Congratulations on the new position. That is excellent news. I am sure you will perform very well.

Example response pair prediction:
Prediction: Positive



Output format (follow exactly, do not output any extra text):
Prediction: <Positive or Negative>



Given these instructions, predit whether the feature is positive or negative for the following response pair:



"""

In [ ]:
def ground_truth_from_score(score):
    return "Positive" if score > 0 else "Negative"

def parse_prediction_response(prediction_response):
    if "Positive" in prediction_response:
        return "Positive"
    elif "Negative" in prediction_response:
        return "Negative"
    else:
        None

In [ ]:
from openai import OpenAI, AsyncOpenAI, PermissionDeniedError
client = OpenAI()
aclient = AsyncOpenAI()

In [ ]:
import asyncio

def get_feature_label(training_set):
    labeling_prompt = labeling_instructions
    middle_index = len(training_set) // 2
    for i, sample in enumerate(training_set[middle_index:]):
        labeling_prompt += f"\nPositive Example {i + 1}:\n"
        #labeling_prompt += f"Score: {sample['score']}\n"
        labeling_prompt += f"Response One:\n{sample['chosen']}\n"
        labeling_prompt += f"Response Two:\n{sample['rejected']}\n"
        labeling_prompt += "-----"    
    for i, sample in enumerate(training_set[:middle_index]):
        labeling_prompt += f"\nNegative Example {i + 1}:\n"
        #labeling_prompt += f"Score: {sample['score']}\n"
        labeling_prompt += f"Response One:\n{sample['chosen']}\n"
        labeling_prompt += f"Response Two:\n{sample['rejected']}\n"
        labeling_prompt += "-----"
    response = client.responses.create(
        model="gpt-5",
        reasoning={"effort": "medium"},
        input=labeling_prompt
    )
    return response.output_text

async def get_prediction(activating_example, prediction_prompt, semaphore):
    async with semaphore:
        prompt = (
            prediction_prompt
            + f"\nResponse One:\n{activating_example['chosen']}\n"
            + f"Response Two:\n{activating_example['rejected']}\n"
        )
        try:
            response = await aclient.responses.create(
                model="gpt-5",
                reasoning={"effort": "medium"},
                input=prompt
            )
            prediction = parse_prediction_response(response.output_text)
            if prediction is None:
                print(f"Unparseable prediction: {response.output_text}")
                return None
            ground_truth = ground_truth_from_score(activating_example["score"])
            return prediction == ground_truth
        except PermissionDeniedError:
            print(f"Permission denied during prediction")
            return None

async def evaluate_feature_description(feature_id, feature_description, held_out_set):
    prediction_prompt = prediction_instructions + f"\nFeature Description:\n{feature_description}\n"
    semaphore = asyncio.Semaphore(50) 
    tasks = [
        get_prediction(activating_example, prediction_prompt, semaphore) 
        for activating_example in held_out_set
    ]
    print(f"Feature {feature_id}: Evaluating {len(held_out_set)} examples in parallel...")
    results = await asyncio.gather(*tasks)
    results = [result for result in results if result is not None]
    correct_count = sum(results)
    total_count = len(results)
    accuracy = correct_count / total_count
    feature_scores[feature_id] = {
        "accuracy": accuracy,
        "correct_count": correct_count,
        "total_count": total_count,
        "label": feature_description,
    }
    print(f"Feature {feature_id}: accuracy={accuracy:.3f} ({correct_count}/{total_count})")

async def process_feature(feature_id):
    if mode == "topk":
        if not loader.has_sufficient_data(feature_id, 2 * k + held_out_count):
            print(f"Skipping feature {feature_id}: insufficient data.")
            return
        training_set, held_out_set = loader.get_top_examples(feature_id, k, held_out_count, seed, random_heldouts)
    elif mode == "range":
        if not loader.has_sufficient_data(feature_id, quantile_count * samples_per_quantile + held_out_count):
            print(f"Skipping feature {feature_id}: insufficient data.")
            return
        training_set, held_out_set = loader.get_quantile_examples(feature_id, quantile_count, samples_per_quantile, held_out_count, seed)
    print(f"Feature {feature_id}: Generating feature description...")
    try:
        feature_description = get_feature_label(training_set)
    except PermissionDeniedError:
        print(f"Feature {feature_id}: Skipping due to permission error.")
        return
    print(f"Feature {feature_id}: Evaluating feature description...")
    await evaluate_feature_description(feature_id, feature_description, held_out_set)

feature_scores = {}

mode="topk"
quantile_count = 5
samples_per_quantile = 2
k = 5
held_out_count = 50
random_heldouts = False
seed = 42

for i in range(20, 32):
    await process_feature(i)

In [ ]:
feature_scores

In [ ]:
import html
from IPython.display import HTML

def render_pair_html(feature_id, activating_examples):
    css = """
    <style>
        .pair-container { font-family: sans-serif; margin-bottom: 20px; border: 1px solid #444; background: #1e1e1e; color: #ddd; }
        .header { background: #333; padding: 8px 12px; font-weight: bold; display: flex; justify-content: space-between; }
        .score-pill { padding: 2px 8px; border-radius: 10px; font-family: monospace; }
        .content { display: flex; width: 100%; }
        .col { flex: 1; padding: 15px; overflow-wrap: break-word; white-space: pre-wrap; }
        .col-chosen { border-right: 1px solid #444; background: #1a2a1a; }
        .col-rejected { background: #2a1a1a; }
        .label { display: block; font-size: 0.8em; text-transform: uppercase; margin-bottom: 8px; font-weight: bold; letter-spacing: 1px; }
        .label-c { color: #8f8; }
        .label-r { color: #f88; }
        .txt { font-family: 'Consolas', 'Monaco', monospace; font-size: 0.9em; line-height: 1.4; }
    </style>
    """
    html_content = [css]
    html_content.append(f"<h3>Feature {feature_id} - Top Activating Pairs</h3>")
    for activating_example in activating_examples:
        score = activating_example['score']
        background = f"rgba(0, 150, 255, {min(abs(score)/10, 1.0)})" if score > 0 else f"rgba(255, 50, 50, {min(abs(score)/10, 1.0)})"
        html_content.append(f"""
        <div class="pair-container">
            <div class="header">
                <span>Example</span>
                <span class="score-pill" style="background: {background}; color: white;">Difference Score: {score:.4f}</span>
            </div>
            <div class="content">
                <div class="col col-chosen">
                    <span class="label label-c">Chosen Response</span>
                    <div class="txt">{html.escape(activating_example['chosen'])}</div>
                </div>
                <div class="col col-rejected">
                    <span class="label label-r">Rejected Response</span>
                    <div class="txt">{html.escape(activating_example['rejected'])}</div>
                </div>
            </div>
        </div>
        """)
    return "".join(html_content)

def interact_with_features(loader, features_list):
    current_index = [0]
    button_previous = widgets.Button(description="Previous")
    button_next = widgets.Button(description="Next")
    feature_label = widgets.Label(value=f"Feature: {features_list[0]}")
    output = widgets.Output()
    def show():
        output.clear_output(wait=True)
        feature_id = features_list[current_index[0]]
        feature_label.value = f"Feature: {feature_id} ({current_index[0]+1}/{len(features_list)})"
        with output:
            if not loader.has_sufficient_data(feature_id, 2 * k):
                print(f"Skipping feature {feature_id}: insufficient data.")
                return
            decoded_training_set = loader.get_top_examples(feature_id, k=10, held_out_count=0, seed=42)
            middle_index = len(decoded_training_set) // 2
            top_k = decoded_training_set[middle_index:]
            bottom_k = decoded_training_set[:middle_index]
            top_k.sort(key=lambda x: x['score'], reverse=True)
            bottom_k.sort(key=lambda x: x['score'], reverse=False)
            combined = top_k + bottom_k
            display(HTML(render_pair_html(feature_id, combined)))
    def on_next(b):
        current_index[0] = (current_index[0] + 1) % len(features_list)
        show()
    button_next.on_click(on_next)
    def on_prev(b):
        current_index[0] = (current_index[0] - 1) % len(features_list)
        show()
    button_previous.on_click(on_prev)
    display(widgets.VBox([widgets.HBox([button_previous, feature_label, button_next]), output]))
    show()

k = 10
FEATURES_TO_BROWSE = range(32) 
interact_with_features(loader, FEATURES_TO_BROWSE)

In [ ]:
# %%
# -----------------------------------------------------------------------------
# Visualization Logic (Quantile Version)
# -----------------------------------------------------------------------------
def render_pair_html(feature_id, examples, total_quantiles):
    css = """
    <style>
        .pair-container { font-family: sans-serif; margin-bottom: 20px; border: 1px solid #444; background: #1e1e1e; color: #ddd; }
        .header { background: #333; padding: 8px 12px; font-weight: bold; display: flex; justify-content: space-between; align-items: center; }
        .meta-info { display: flex; gap: 10px; align-items: center; }
        .score-pill { padding: 2px 8px; border-radius: 4px; font-family: monospace; font-size: 0.9em; }
        .bucket-pill { background: #555; padding: 2px 8px; border-radius: 4px; font-size: 0.8em; color: #fff; }
        .content { display: flex; width: 100%; }
        .col { flex: 1; padding: 15px; overflow-wrap: break-word; white-space: pre-wrap; }
        .col-chosen { border-right: 1px solid #444; background: #1a2a1a; }
        .col-rejected { background: #2a1a1a; }
        .label { display: block; font-size: 0.8em; text-transform: uppercase; margin-bottom: 8px; font-weight: bold; letter-spacing: 1px; }
        .label-c { color: #8f8; }
        .label-r { color: #f88; }
        .txt { font-family: 'Consolas', 'Monaco', monospace; font-size: 0.9em; line-height: 1.4; }
    </style>
    """
    html_content = [css]
    html_content.append(f"<h3>Feature {feature_id} - Quantile Samples</h3>")
    
    for ex in examples:
        score = ex['score']
        bucket = ex.get('bucket_index', '?')
        
        # Color intensity based on score magnitude
        alpha = min(abs(score) / 5.0, 1.0) # Adjust divisor to sensitivity
        bg_color = f"rgba(0, 150, 255, {alpha})" if score > 0 else f"rgba(255, 50, 50, {alpha})"
        
        html_content.append(f"""
        <div class="pair-container">
            <div class="header">
                <div class="meta-info">
                    <span class="bucket-pill">Quantile {bucket}/{total_quantiles}</span>
                    <span>Example</span>
                </div>
                <span class="score-pill" style="background: {bg_color}; color: white;">Score: {score:.4f}</span>
            </div>
            <div class="content">
                <div class="col col-chosen">
                    <span class="label label-c">Chosen Response</span>
                    <div class="txt">{html.escape(ex['chosen'])}</div>
                </div>
                <div class="col col-rejected">
                    <span class="label label-r">Rejected Response</span>
                    <div class="txt">{html.escape(ex['rejected'])}</div>
                </div>
            </div>
        </div>
        """)
    return "".join(html_content)

def interact_with_features(loader, features_list, n_quantiles=10, m_samples=1):
    current_idx = [0]
    out = widgets.Output()
    btn_prev = widgets.Button(description="Previous")
    btn_next = widgets.Button(description="Next")
    lbl_feat = widgets.Label(value=f"Feature: {features_list[0]}")
    
    def show():
        out.clear_output(wait=True)
        feat_id = features_list[current_idx[0]]
        lbl_feat.value = f"Feature: {feat_id} ({current_idx[0]+1}/{len(features_list)})"
        
        with out:
            # 1. Get Quantile Examples
            # Returns [Low Score -> High Score]
            examples = loader.get_quantile_examples(
                feat_id, 
                n_quantiles=n_quantiles, 
                m_samples_per_bucket=m_samples
            )
            
            if not examples:
                print("No activations found for this feature.")
            else:
                # 2. Reverse list to show Highest Scores (Top Quantiles) first
                examples.reverse()
                
                # 3. Render
                display(HTML(render_pair_html(feat_id, examples, n_quantiles)))

    def on_next(b):
        current_idx[0] = (current_idx[0] + 1) % len(features_list)
        show()
    def on_prev(b):
        current_idx[0] = (current_idx[0] - 1) % len(features_list)
        show()
        
    btn_next.on_click(on_next)
    btn_prev.on_click(on_prev)
    
    # Layout
    ctrl_box = widgets.HBox([btn_prev, lbl_feat, btn_next])
    display(widgets.VBox([ctrl_box, out]))
    show()

# -----------------------------------------------------------------------------
# Run Visualization
# -----------------------------------------------------------------------------
# n_quantiles=10, m_samples=1 => Shows ~10 examples ranging from Max Activation down to Min Activation
FEATURES_TO_BROWSE = range(32) 
interact_with_features(loader, FEATURES_TO_BROWSE, n_quantiles=10, m_samples=2)

In [ ]:
# %%
def save_feature_to_txt(loader, feature_id, n_pos, m_neg, filename):
    print(f"Generating report for Feature {feature_id}...")
    print(f"Retrieving top {n_pos} positive and top {m_neg} negative examples...")

    # Fetch Data
    pos_examples = loader.get_top_examples(feature_id, k=n_pos, largest=True)
    neg_examples = loader.get_top_examples(feature_id, k=m_neg, largest=False)
    
    # Filter strictly (pos > 0, neg < 0)
    pos_examples = [ex for ex in pos_examples if ex['score'] > 0]
    neg_examples = [ex for ex in neg_examples if ex['score'] < 0]

    with open(filename, "w", encoding="utf-8") as f:
        # Header
        f.write(f"Positive feature activations:\n\n")
        
        if not pos_examples:
            f.write("(No positive activations found)\n\n")
        
        for i, ex in enumerate(pos_examples, 1):
            f.write(f"Preference pair {i} (activation: {ex['score']:.4f})\n\n")
            f.write("Chosen Response\n\n")
            f.write(f"{ex['chosen'].strip()}\n\n")
            f.write("Rejected Response\n\n")
            f.write(f"{ex['rejected'].strip()}\n\n")
            f.write("-" * 50 + "\n\n")
            
        f.write("\n" + "=" * 50 + "\n\n")
        
        f.write(f"Negative feature activations:\n\n")
        
        if not neg_examples:
            f.write("(No negative activations found)\n\n")
            
        for i, ex in enumerate(neg_examples, 1):
            f.write(f"Preference pair {i} (activation: {ex['score']:.4f})\n\n")
            f.write("Chosen Response\n\n")
            f.write(f"{ex['chosen'].strip()}\n\n")
            f.write("Rejected Response\n\n")
            f.write(f"{ex['rejected'].strip()}\n\n")
            f.write("-" * 50 + "\n\n")
            
    print(f"Successfully saved to {filename}")

# -----------------------------------------------------------------------------
# SETTINGS
# -----------------------------------------------------------------------------
FEATURE_TO_SAVE = 3           # The specific feature ID you want to save
N_POS = 5                     # Number of top positive examples
M_NEG = 5                     # Number of top negative examples
OUTPUT_FILENAME = f"feature_{FEATURE_TO_SAVE}_report.txt"

save_feature_to_txt(loader, FEATURE_TO_SAVE, N_POS, M_NEG, OUTPUT_FILENAME)

In [ ]:
import torch
from pathlib import Path
from safetensors.torch import load_file

CACHE_DIR = Path("results/hhrlhf_diff_32/norm")

def inspect_cache_statistics(cache_dir):
    files = sorted(list(cache_dir.glob("*.safetensors")))
    print(f"Found {len(files)} files.")
    
    # Store counts per feature
    global_pos_counts = {}
    global_neg_counts = {}
    
    print(f"{'Filename':<30} | {'Range':<15} | {'Acts':<8} | {'Min Loc':<8} | {'Max Loc':<8}")
    print("-" * 85)

    for file_path in files:
        # Parse offset from filename: start_end.safetensors
        try:
            name_parts = file_path.stem.split("_")
            start_offset = int(name_parts[0])
            end_offset = int(name_parts[1])
        except:
            print(f"⚠️ Could not parse offset from {file_path.name}. Assuming offset 0.")
            start_offset = 0

        # Load data
        with torch.no_grad():
            try:
                data = load_file(file_path)
                locs = data["locations"].to(torch.int64)
                acts = data["activations"].to(torch.float32)
            except Exception as e:
                print(f"Error loading {file_path.name}: {e}")
                continue
            
            if locs.numel() == 0:
                print(f"{file_path.name:<30} | {start_offset}-{end_offset:<8} | 0        | -        | -")
                continue

            local_ids = locs[:, 2]
            global_ids = local_ids + start_offset
            
            print(f"{file_path.name:<30} | {start_offset}-{end_offset:<8} | {len(locs):<8} | {local_ids.min().item():<8} | {local_ids.max().item():<8}")

            # Count per-feature positive & negative activation
            for gid, act in zip(global_ids, acts):
                gid = gid.item()
                a = act.item()
                
                if a > 0:
                    global_pos_counts[gid] = global_pos_counts.get(gid, 0) + 1
                elif a < 0:
                    global_neg_counts[gid] = global_neg_counts.get(gid, 0) + 1

    print("-" * 85)
    all_features = sorted(set(global_pos_counts.keys()) | set(global_neg_counts.keys()))
    print(f"Total Unique Features Fired: {len(all_features)}")

    # Summary for your first 32 features
    print("\n--- Positive vs Negative Firing for Features 0–31 ---")
    for i in range(32):
        pos = global_pos_counts.get(i, 0)
        neg = global_neg_counts.get(i, 0)

        if pos == 0 and neg == 0:
            print(f"Feature {i:2d}: ❌ DEAD (0 fires)")
        else:
            print(f"Feature {i:2d}: +{pos:5d}  /  -{neg:5d}")

inspect_cache_statistics(CACHE_DIR)
